# Regenerate the final report figures

Run after the clean and joint-table notebooks from the repository root. Generated figures are written to the ignored `outputs/report_figures/` directory.


In [ ]:
# =========================================================
# Regenerate all report figures (fixed arrow directions)
# =========================================================


from pathlib import Path
import os

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/project_clean.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the cloned project directory.")
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
import textwrap
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from pyspark.sql import SparkSession, functions as F

# ---------------------------------------------------------
# Spark
# ---------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("Generate Fixed Report Figures")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
FLIGHT_RAW_PATH = str(DATA_ROOT / "raw" / "Flights")
WEATHER_RAW_PATH = str(DATA_ROOT / "raw" / "Weather")

FLIGHT_CLEAN_PATH = str(DATA_ROOT / "clean" / "Flights")
WEATHER_CLEAN_PATH = str(DATA_ROOT / "clean" / "Weather")
JT_OUTPUT_PATH = str(DATA_ROOT / "jt")

FIG_DIR = str(OUTPUT_ROOT / "report_figures")
os.makedirs(FIG_DIR, exist_ok=True)

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def save_fig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()

def make_canvas(figsize=(14, 4), title=None):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    if title:
        ax.text(0.5, 0.95, title, ha="center", va="top", fontsize=18)
    return fig, ax

def add_box(ax, x, y, w, h, text, fontsize=11):
    # x, y are lower-left coordinates in [0,1]
    rect = Rectangle((x, y), w, h, fill=False, linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fontsize)
    return {"x": x, "y": y, "w": w, "h": h}

def anchor(box, side):
    if side == "left":
        return (box["x"], box["y"] + box["h"] / 2)
    if side == "right":
        return (box["x"] + box["w"], box["y"] + box["h"] / 2)
    if side == "top":
        return (box["x"] + box["w"] / 2, box["y"] + box["h"])
    if side == "bottom":
        return (box["x"] + box["w"] / 2, box["y"])
    raise ValueError("side must be left/right/top/bottom")

def connect(ax, box1, side1, box2, side2, shrinkA=2, shrinkB=2):
    x1, y1 = anchor(box1, side1)
    x2, y2 = anchor(box2, side2)
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(
            arrowstyle="->",
            lw=1.5,
            shrinkA=shrinkA,
            shrinkB=shrinkB,
            connectionstyle="arc3,rad=0.0"
        )
    )

def parquet_exists(path):
    try:
        spark.read.parquet(path).limit(1).count()
        return True
    except:
        return False

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
flights_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{FLIGHT_RAW_PATH}/*")
)

weather_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(f"{WEATHER_RAW_PATH}/*")
)

flights_clean = spark.read.parquet(FLIGHT_CLEAN_PATH)
weather_clean = spark.read.parquet(WEATHER_CLEAN_PATH)

jt_dfs = []
for m in [4, 6, 9, 11]:
    p = f"{JT_OUTPUT_PATH}/year=2013/month={m}"
    if parquet_exists(p):
        jt_dfs.append(spark.read.parquet(p).withColumn("run_month", F.lit(m)))

jt_all = None
if jt_dfs:
    jt_all = jt_dfs[0]
    for d in jt_dfs[1:]:
        jt_all = jt_all.unionByName(d)


In [ ]:

# =========================================================
# FIG 01 overall pipeline
# =========================================================
fig, ax = make_canvas(figsize=(14, 4), title="Overall Pipeline")

raw_flights = add_box(ax, 0.02, 0.35, 0.16, 0.22, "Raw Flights CSV", 11)
raw_weather = add_box(ax, 0.23, 0.35, 0.16, 0.22, "Raw Weather CSV", 11)
mapping     = add_box(ax, 0.44, 0.35, 0.16, 0.22, "WBAN-Airport-TimeZone\nMapping", 11)

clean_f     = add_box(ax, 0.66, 0.54, 0.14, 0.18, "Clean Flights\n(Parquet)", 11)
clean_w     = add_box(ax, 0.66, 0.18, 0.14, 0.18, "Clean Weather\n(Parquet)", 11)

jt_box      = add_box(ax, 0.86, 0.35, 0.14, 0.22, "Monthly Joint Table\n(JT)", 11)

connect(ax, raw_flights, "right", clean_f, "left")
connect(ax, raw_weather, "right", clean_w, "left")
connect(ax, mapping, "right", clean_f, "left")
connect(ax, mapping, "right", clean_w, "left")
connect(ax, clean_f, "right", jt_box, "left")
connect(ax, clean_w, "right", jt_box, "left")

save_fig(f"{FIG_DIR}/fig01_overall_pipeline.png")


In [ ]:

# =========================================================
# FIG 02 data source relationship
# =========================================================
fig, ax = make_canvas(figsize=(12, 5), title="Relationship Among Data Sources")

flights_box = add_box(ax, 0.08, 0.58, 0.20, 0.18, "Flights\nAirport-based", 11)
weather_box = add_box(ax, 0.08, 0.16, 0.20, 0.18, "Weather\nWBAN-based", 11)
map_box     = add_box(ax, 0.40, 0.37, 0.24, 0.22, "Mapping Table\nAirportID ↔ WBAN\nTimeZone", 11)
join_box    = add_box(ax, 0.76, 0.37, 0.16, 0.22, "Joinable Data", 11)

connect(ax, flights_box, "right", map_box, "left")
connect(ax, weather_box, "right", map_box, "left")
connect(ax, map_box, "right", join_box, "left")

save_fig(f"{FIG_DIR}/fig02_data_sources_relationship.png")


In [ ]:

# =========================================================
# FIG 03 flight cleaning counts
# =========================================================
raw_count = flights_raw.count()

flights_1 = flights_raw.drop("_c12")
flights_2 = (
    flights_1
    .withColumn("FL_DATE", F.to_date("FL_DATE", "yyyy-MM-dd"))
    .withColumn("CANCELLED", F.col("CANCELLED").cast("double"))
    .withColumn("DIVERTED", F.col("DIVERTED").cast("double"))
    .withColumn("CRS_ELAPSED_TIME", F.col("CRS_ELAPSED_TIME").cast("double"))
    .withColumn("CRS_DEP_TIME", F.col("CRS_DEP_TIME").cast("string"))
)
count_after_cast = flights_2.count()

flights_3 = (
    flights_2
    .filter(F.col("CANCELLED") == 0.0)
    .filter(F.col("DIVERTED") == 0.0)
    .filter(F.col("CRS_DEP_TIME").isNotNull())
    .filter(F.col("CRS_ELAPSED_TIME").isNotNull())
)
count_after_filter = flights_3.count()

flights_4 = (
    flights_3
    .withColumn("CRS_DEP_TIME_4", F.lpad(F.regexp_replace(F.col("CRS_DEP_TIME"), r"[^0-9]", ""), 4, "0"))
    .withColumn("dep_hour", F.substring("CRS_DEP_TIME_4", 1, 2).cast("int"))
    .withColumn("dep_minute", F.substring("CRS_DEP_TIME_4", 3, 2).cast("int"))
    .filter(F.col("dep_hour").between(0, 23))
    .filter(F.col("dep_minute").between(0, 59))
)
count_after_valid_time = flights_4.count()

count_final = flights_clean.count()

labels = [
    "Raw CSV",
    "After Cast",
    "After Filter\nCancelled/Diverted",
    "After Valid\nDep Time",
    "Final Clean"
]
values = [
    raw_count,
    count_after_cast,
    count_after_filter,
    count_after_valid_time,
    count_final
]

plt.figure(figsize=(12, 5))
plt.bar(labels, values)
plt.title("Flight Row Counts Across Cleaning Steps")
plt.ylabel("Row count")
for i, v in enumerate(values):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=10)
save_fig(f"{FIG_DIR}/fig03_flight_cleaning_counts.png")


In [ ]:

# =========================================================
# FIG 04 scheduled departure timestamp construction
# =========================================================
fig, ax = make_canvas(figsize=(14, 4), title="Scheduled Departure Timestamp Construction")

b1 = add_box(ax, 0.03, 0.32, 0.18, 0.22, "FL_DATE", 11)
b2 = add_box(ax, 0.27, 0.32, 0.18, 0.22, "CRS_DEP_TIME", 11)
b3 = add_box(ax, 0.51, 0.32, 0.18, 0.22, "Pad to 4 digits\nSplit hour/minute", 11)
b4 = add_box(ax, 0.75, 0.32, 0.22, 0.22, "scheduled_dep_ts", 11)

connect(ax, b1, "right", b3, "left")
connect(ax, b2, "right", b3, "left")
connect(ax, b3, "right", b4, "left")

save_fig(f"{FIG_DIR}/fig04_scheduled_dep_ts_construction.png")


In [ ]:

# =========================================================
# FIG 05 arrival timestamp and time zone logic
# =========================================================
fig, ax = make_canvas(figsize=(14, 5), title="Arrival Timestamp and Time Zone Logic")

d1 = add_box(ax, 0.03, 0.38, 0.18, 0.20, "scheduled_dep_ts", 11)
d2 = add_box(ax, 0.27, 0.38, 0.18, 0.20, "CRS_ELAPSED_TIME", 11)
d3 = add_box(ax, 0.51, 0.38, 0.20, 0.20, "scheduled_arr_ts\n_origin_clock", 11)

tz_o = add_box(ax, 0.80, 0.62, 0.14, 0.14, "origin_tz", 11)
tz_d = add_box(ax, 0.80, 0.22, 0.14, 0.14, "dest_tz", 11)
tz_x = add_box(ax, 0.80, 0.42, 0.16, 0.14, "tz_diff_hours", 11)
d4   = add_box(ax, 0.80, 0.02, 0.16, 0.14, "scheduled_arr_ts\n_dest_local", 10)

connect(ax, d1, "right", d3, "left")
connect(ax, d2, "right", d3, "left")
connect(ax, d3, "right", tz_x, "left")
connect(ax, tz_o, "bottom", tz_x, "top")
connect(ax, tz_d, "top", tz_x, "bottom")
connect(ax, tz_x, "bottom", d4, "top")

save_fig(f"{FIG_DIR}/fig05_arrival_ts_timezone_logic.png")


In [ ]:

# =========================================================
# FIG 06 clean flight schema
# =========================================================
fig, ax = make_canvas(figsize=(10, 8), title="Core Schema of Clean Flight Table")

schema_text = """
flight_id
FL_DATE
flight_year, flight_month, flight_yyyymm
OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM
ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID
CRS_DEP_TIME, dep_hour, dep_minute
CRS_ELAPSED_TIME
scheduled_dep_ts
origin_tz, dest_tz, tz_diff_hours
scheduled_arr_ts_origin_clock
scheduled_arr_ts_dest_local
ARR_DELAY_NEW, arr_delay_min
WEATHER_DELAY, NAS_DELAY
has_weather_delay, has_nas_delay
is_delayed_15, is_delayed_60
""".strip()

add_box(ax, 0.15, 0.10, 0.70, 0.78, schema_text, 10)
save_fig(f"{FIG_DIR}/fig06_flight_output_schema.png")


In [ ]:

# =========================================================
# FIG 07 weather obs timestamp construction
# =========================================================
fig, ax = make_canvas(figsize=(14, 4), title="Weather Observation Timestamp Construction")

w1 = add_box(ax, 0.03, 0.32, 0.18, 0.22, "Date", 11)
w2 = add_box(ax, 0.25, 0.32, 0.18, 0.22, "Time", 11)
w3 = add_box(ax, 0.47, 0.32, 0.20, 0.22, "Pad to 4 digits\nSplit hour/minute", 11)
w4 = add_box(ax, 0.72, 0.32, 0.22, 0.22, "obs_ts", 11)

connect(ax, w1, "right", w3, "left")
connect(ax, w2, "right", w3, "left")
connect(ax, w3, "right", w4, "left")

save_fig(f"{FIG_DIR}/fig07_weather_obs_ts_construction.png")


In [ ]:

# =========================================================
# FIG 08 weather cleaning pipeline
# =========================================================
fig, ax = make_canvas(figsize=(16, 4), title="Weather Cleaning and Airport Mapping Pipeline")

p1 = add_box(ax, 0.02, 0.32, 0.13, 0.20, "Raw Weather CSV", 10)
p2 = add_box(ax, 0.18, 0.32, 0.14, 0.20, "Trim strings\nReplace '' / M with null", 10)
p3 = add_box(ax, 0.36, 0.32, 0.12, 0.20, "Keep selected columns", 10)
p4 = add_box(ax, 0.52, 0.32, 0.16, 0.20, "Build obs_ts\nClean numeric fields", 10)
p5 = add_box(ax, 0.72, 0.32, 0.13, 0.20, "Join WBAN mapping", 10)
p6 = add_box(ax, 0.88, 0.32, 0.10, 0.20, "Clean Weather Parquet", 10)

connect(ax, p1, "right", p2, "left")
connect(ax, p2, "right", p3, "left")
connect(ax, p3, "right", p4, "left")
connect(ax, p4, "right", p5, "left")
connect(ax, p5, "right", p6, "left")

save_fig(f"{FIG_DIR}/fig08_weather_cleaning_pipeline.png")


In [ ]:

# =========================================================
# FIG 09 buffered weather read logic
# =========================================================
fig, ax = make_canvas(figsize=(12, 4), title="Buffered Weather Read Logic")

m1 = add_box(ax, 0.10, 0.42, 0.18, 0.18, "Prev Month\nWeather", 11)
m2 = add_box(ax, 0.41, 0.42, 0.18, 0.18, "Current Month\nWeather", 11)
m3 = add_box(ax, 0.72, 0.42, 0.18, 0.18, "Next Month\nWeather", 11)
m4 = add_box(ax, 0.41, 0.08, 0.18, 0.16, "Buffered Weather\nfor Join", 11)

connect(ax, m1, "bottom", m4, "top")
connect(ax, m2, "bottom", m4, "top")
connect(ax, m3, "bottom", m4, "top")

save_fig(f"{FIG_DIR}/fig09_weather_buffer_logic.png")


In [ ]:

# =========================================================
# FIG 10 weather matching logic
# =========================================================
fig, ax = make_canvas(figsize=(14, 5), title="Flight Target Time and Weather Matching Logic")

q1 = add_box(ax, 0.04, 0.36, 0.18, 0.20, "flight_id\nairport_id\nbase_ts", 11)
q2 = add_box(ax, 0.28, 0.36, 0.16, 0.20, "Cross join\nlag_hour = 0..12", 11)
q3 = add_box(ax, 0.50, 0.36, 0.18, 0.20, "target_ts =\nbase_ts - lag*1h", 11)
q4 = add_box(ax, 0.74, 0.36, 0.18, 0.20, "Join weather by\nairport_id", 11)
q5 = add_box(ax, 0.74, 0.08, 0.18, 0.16, "Keep nearest obs\nwithin 1 hour", 11)

connect(ax, q1, "right", q2, "left")
connect(ax, q2, "right", q3, "left")
connect(ax, q3, "right", q4, "left")
connect(ax, q4, "bottom", q5, "top")

save_fig(f"{FIG_DIR}/fig10_weather_matching_logic.png")


In [ ]:

# =========================================================
# FIG 11 final JT structure
# =========================================================
fig, ax = make_canvas(figsize=(12, 5), title="Structure of Final Monthly Joint Table")

j1 = add_box(ax, 0.06, 0.35, 0.20, 0.18, "flights_m\n(main flight table)", 11)
j2 = add_box(ax, 0.38, 0.60, 0.22, 0.14, "wo_wide\n(origin weather wide)", 11)
j3 = add_box(ax, 0.38, 0.16, 0.22, 0.14, "wd_wide\n(destination weather wide)", 11)
j4 = add_box(ax, 0.74, 0.35, 0.20, 0.18, "jt_month\n(final joint table)", 11)

connect(ax, j1, "right", j4, "left")
connect(ax, j2, "right", j4, "top")
connect(ax, j3, "right", j4, "bottom")

save_fig(f"{FIG_DIR}/fig11_final_jt_structure.png")


In [ ]:

# =========================================================
# FIG 12 code structure
# =========================================================
fig, ax = make_canvas(figsize=(14, 5), title="Code Structure and Output Directories")

c1 = add_box(ax, 0.04, 0.55, 0.18, 0.16, "Part 1:\nFlight Cleaning", 11)
c2 = add_box(ax, 0.04, 0.28, 0.18, 0.16, "Part 2:\nWeather Cleaning", 11)
c3 = add_box(ax, 0.35, 0.42, 0.18, 0.16, "Part 3:\nJT Construction", 11)

o1 = add_box(ax, 0.68, 0.62, 0.20, 0.12, "clean/Flights", 10)
o2 = add_box(ax, 0.68, 0.40, 0.20, 0.12, "clean/Weather", 10)
o3 = add_box(ax, 0.68, 0.18, 0.20, 0.12, "jt/year=2013/month=...", 10)

connect(ax, c1, "right", o1, "left")
connect(ax, c2, "right", o2, "left")
connect(ax, c1, "right", c3, "left")
connect(ax, c2, "right", c3, "left")
connect(ax, c3, "right", o3, "left")

save_fig(f"{FIG_DIR}/fig12_code_structure.png")


In [ ]:
# =========================================================
# FIG 13 future work
# =========================================================
fig, ax = make_canvas(figsize=(13, 6), title="Current Achievements, Limitations, and Future Directions")

# Top center: achieved result
f0 = add_box(
    ax, 0.33, 0.72, 0.34, 0.16,
    "Current Achievements\n\n"
    "- Flight data cleaned and structured\n"
    "- Weather data cleaned and mapped to airports\n"
    "- Monthly joint tables successfully constructed",
    10
)

# Bottom left: limitations
f1 = add_box(
    ax, 0.07, 0.30, 0.34, 0.24,
    "Current Limitations\n\n"
    "- Only selected months were run in the JT stage\n"
    "- Weather matching keeps only the nearest record within 1 hour\n"
    "- Runtime cost is high in weather matching, reshaping, and writing\n"
    "- Downstream prediction is only validated with a simple baseline",
    10
)

# Bottom right: future work
f2 = add_box(
    ax, 0.59, 0.30, 0.34, 0.24,
    "Future Extensions\n\n"
    "- Run more months or larger time ranges\n"
    "- Analyze missing rates after weather matching\n"
    "- Compare alternative weather aggregation strategies\n"
    "- Train and evaluate stronger prediction models\n"
    "- Further optimize Spark execution and output stages",
    10
)

# Middle small connector / interpretation
f3 = add_box(
    ax, 0.38, 0.08, 0.24, 0.11,
    "Overall Message\n\n"
    "The data-preparation pipeline is complete,\n"
    "but scalability and modeling can be further improved.",
    10
)

connect(ax, f0, "bottom", f1, "top")
connect(ax, f0, "bottom", f2, "top")
connect(ax, f1, "bottom", f3, "top")
connect(ax, f2, "bottom", f3, "top")

save_fig(f"{FIG_DIR}/fig13_future_work.png")

print("Saved:", f"{FIG_DIR}/fig13_future_work.png")